In [ ]:
# Cell 1: Imports
import numpy as np
import matplotlib.pyplot as plt
plt.tight_layout()
import time
from scipy.optimize import minimize

<Figure size 640x480 with 0 Axes>

In [ ]:
# Cell 2: Class Definitions

class FermatWeberTrustRegionBFGS:
    """
    Implementación del método BFGS con Región de Confianza (Trust Region BFGS)
    para el Problema de Fermat-Weber, utilizando la estrategia Dogleg.
    """

    def __init__(self, points, weights=None, epsilon=1e-8, max_iter=1000):
        self.points = np.array(points)
        self.m = len(points)
        self.n = len(points[0])

        if weights is None:
            self.weights = np.ones(self.m)
        else:
            self.weights = np.array(weights)

        self.epsilon = epsilon
        self.max_iter = max_iter

    def objective_function(self, x):
        """Calcula la suma ponderada de distancias (Función de Fermat-Weber)."""
        diff = self.points - x
        dists = np.linalg.norm(diff, axis=1)
        return np.dot(self.weights, dists)

    def gradient(self, x):
        """Calcula el gradiente con protección para singularidades (puntos)."""
        grad = np.zeros(self.n)
        for i in range(self.m):
            diff = x - self.points[i]
            distance = np.linalg.norm(diff)
            # Pequeña tolerancia para evitar división por cero en el punto
            if distance > 1e-12:
                grad += self.weights[i] * diff / distance
        return grad

    def dogleg_step(self, grad, B, delta):
        """
        Calcula el paso p utilizando el método Dogleg dentro de la región de
        confianza de radio delta.
        """
        # --- 1. Punto de Cauchy (p_U) ---
        curvature = grad.T @ B @ grad

        if curvature <= 1e-10:
            norm_g = np.linalg.norm(grad)
            if norm_g == 0: return np.zeros_like(grad)
            # Si la curvatura es ~0, avanzamos lo más posible en la dirección del gradiente
            return - (delta / norm_g) * grad

        tau_c = (grad.T @ grad) / curvature
        pU = -tau_c * grad
        pU_norm = np.linalg.norm(pU)

        # Si el paso de Cauchy ya excede el radio, se recorta
        if pU_norm >= delta:
            return (delta / pU_norm) * pU

        # --- 2. Paso de Newton (p_B) ---
        try:
            # Solución del sistema B*p = -grad
            pB = -np.linalg.solve(B, grad)
        except np.linalg.LinAlgError:
            pB = pU # Fallback si B es singular

        pB_norm = np.linalg.norm(pB)

        # Si el paso de Newton cabe, se toma
        if pB_norm <= delta:
            return pB

        # --- 3. Dogleg (Interpolación entre p_U y p_B) ---
        # Resolvemos ||pU + tau * (pB - pU)||^2 = delta^2 para tau en [0, 1]
        pU_pB = pB - pU
        a = pU_pB.T @ pU_pB
        b = 2 * pU.T @ pU_pB
        c = pU.T @ pU - delta**2

        # Usando la fórmula cuadrática para el factor tau
        tau_val = (-b + np.sqrt(max(0, b**2 - 4*a*c))) / (2*a)

        # El paso Dogleg es pU + tau_val * (pB - pU)
        return pU + tau_val * pU_pB

    def solve(self, x0, delta0=1.0, eta=0.1):
        """
        Bucle principal del algoritmo de Región de Confianza.
        :param x0: Punto inicial.
        :param delta0: Radio inicial de la región de confianza.
        :param eta: Umbral para aceptar un paso (0 < eta < 1).
        :return: Solución x y el historial de optimización.
        """
        x = x0.copy()
        delta = delta0 # Radio de la Región de Confianza
        B = np.eye(self.n) # Aproximación inicial del Hessiano

        # --- Historial para análisis ---
        history = {
            'obj': [], 'times': [], 'norm_grad': [],
            'rho': [], 'delta': [], # 'delta' almacena el radio de la región de confianza
            'path': [] # Almacenar la trayectoria (puntos)
        }

        start_time = time.time()
        grad = self.gradient(x)

        for k in range(self.max_iter):
            f_curr = self.objective_function(x)
            grad_norm = np.linalg.norm(grad)

            # --- REGISTRO DEL ESTADO ACTUAL ---
            history['obj'].append(f_curr)
            history['times'].append(time.time() - start_time)
            history['norm_grad'].append(grad_norm)
            history['delta'].append(delta) # Guardar el radio actual
            history['path'].append(x.copy())

            # 1. Verificar convergencia
            if grad_norm < self.epsilon:
                break

            # 2. Obtener paso p mediante Dogleg
            p = self.dogleg_step(grad, B, delta)

            # Si el paso es insignificante, terminamos
            if np.linalg.norm(p) < 1e-12:
                break

            # 3. Evaluar paso
            x_candidate = x + p
            f_candidate = self.objective_function(x_candidate)

            # Reducción Actual (Actual Reduction)
            actual_reduction = f_curr - f_candidate

            # Reducción Predicha (Predicted Reduction) con el modelo cuadrático
            # m(0) - m(p) = -(g^T p + 0.5 * p^T B p)
            predicted_reduction = -(grad.T @ p + 0.5 * p.T @ B @ p)

            # Calculamos rho = Reducción Actual / Reducción Predicha
            rho = actual_reduction / predicted_reduction if predicted_reduction > 1e-12 else -1
            history['rho'].append(rho)

            # 4. Actualización de x y Delta
            if rho > eta:
                # --- PASO ACEPTADO (Actualización de x y BFGS) ---
                x_new = x_candidate
                new_grad = self.gradient(x_new)

                # Variables para la actualización BFGS
                s = x_new - x
                y = new_grad - grad

                # Actualización BFGS (con protección de curvatura positiva)
                sy = s.T @ y
                if sy > 1e-10:
                    # Escalado Inicial (opcional, pero mejora el comportamiento inicial)
                    if k == 0:
                        scale = sy / (y.T @ y) # Escalado según Nocedal & Wright, eq 6.20
                        B = scale * np.eye(self.n)

                    # Actualización BFGS
                    Bs = B @ s
                    term1 = np.outer(Bs, Bs) / (s.T @ Bs)
                    term2 = np.outer(y, y) / sy
                    B = B - term1 + term2

                x = x_new
                grad = new_grad

                # Expandir región si el paso fue excelente
                if rho > 0.75 and np.linalg.norm(p) > 0.8 * delta:
                    delta = min(2.0 * delta, 100.0)
            else:
                # --- PASO RECHAZADO (x y B no cambian) ---
                history['path'].pop() # Eliminar el último punto si fue rechazado
                history['delta'].pop()
                history['obj'].pop()
                history['times'].pop()
                history['norm_grad'].pop()
                history['rho'].pop()

            # Reducir región si el paso fue pobre
            if rho < 0.25:
                delta = 0.5 * delta

            if delta < 1e-10:
                break # Radio de confianza demasiado pequeño

        # Eliminar el último registro de path que se duplica en el bucle
        if len(history['path']) > 0:
            history['path'].pop()

        return x, history

    def solve_path(self, x0, delta0=1.0, eta=0.1):
        """Método auxiliar para obtener solo el path (trayectoria de puntos)."""
        _, history = self.solve(x0, delta0, eta)
        return np.array(history['path'])

In [ ]:
# Cell 3: General Plotting Function (MODIFICADO)

def plot_comparison_with_projection(points, weights, x_w, hist_w, x_b, hist_b, paths_w, paths_b, title, filename_prefix):
    """
    Genera las gráficas de comparación de convergencia (4 paneles de diagnóstico)
    y la gráfica de curvas de nivel de la proyección 2D con trayectorias.
    """
    # 1. Preparar datos
    min_val_w = hist_w['obj'][-1]
    min_val_b = hist_b['obj'][-1]
    # Usar el mínimo entre los dos métodos como aproximación del óptimo f*
    min_val = min(min_val_w, min_val_b)

    err_w = np.array(hist_w['obj']) - min_val + 1e-10
    err_b = np.array(hist_b['obj']) - min_val + 1e-10
    DIMENSION = points.shape[1]

    # --- FIGURE 1: Gráficas de Diagnóstico Detallado (4 Paneles) ---
    # Creación de la figura 2x2
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(f'{title} - Diagnóstico Detallado', fontsize=16, y=1.02)

    ## Panel 1: Convergencia f(x) (Error Log)
    ax1 = axes[0, 0]
    ax1.semilogy(err_w, 'b.-', label='Weiszfeld', linewidth=2)
    ax1.semilogy(err_b, 'r.-', label='BFGS', linewidth=2)
    ax1.set_title(r'Convergencia $f(x)$ (Error Log)')
    ax1.set_xlabel('Iteraciones')
    ax1.set_ylabel(r'Error Log ($f(x) - f^*$)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ## Panel 2: Norma del Gradiente ||g|| (BFGS)
    ax2 = axes[0, 1]
    norm_grad = np.array(hist_b['norm_grad'])
    ax2.semilogy(norm_grad, 'r.-', label='BFGS', linewidth=2)
    # Se utiliza la tolerancia de BFGS del código original (1e-6 o 1e-5/1e-4)
    epsilon_used = 1e-5
    ax2.axhline(epsilon_used, color='gray', linestyle='--', linewidth=1, label=f'Tolerancia ({epsilon_used})')
    ax2.set_title(r'Norma del Gradiente $||\mathbf{g}||$ (BFGS)')
    ax2.set_xlabel('Iteraciones')
    ax2.set_ylabel(r'$||\mathbf{g}||$')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    ## Panel 3: Tamaño de Paso Alpha (BFGS)
    ax3 = axes[1, 0]
    # 'alpha' tiene un elemento menos que el resto del historial, se ajusta el eje x
    iterations = np.arange(len(hist_b['delta']))
    ax3.plot(iterations, hist_b['delta'], 'g.-', label='BFGS', linewidth=2)
    ax3.set_title(r'Radio de region de confianza $\delta$ (BFGS)')
    ax3.set_xlabel('Iteraciones')
    ax3.set_ylabel(r'$\delta$')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    ## Panel 4: Comparativa de Eficiencia (Tiempo/Iters)
    ax4 = axes[1, 1]
    methods = ['Weiszfeld', 'BFGS']
    final_times = [hist_w['times'][-1], hist_b['times'][-1]]
    iter_counts = [len(hist_w['times']), len(hist_b['times'])]

    bars = ax4.bar(methods, final_times, color=['blue', 'red'], alpha=0.6)
    ax4.set_ylabel('Tiempo Total (s)')
    ax4.set_title('Comparativa de Eficiencia')

    for bar, iters in zip(bars, iter_counts):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}s\n({iters} iters)',
                ha='center', va='bottom', fontweight='bold')

    plt.tight_layout(rect=[0, 0.03, 1, 0.98])
    plt.savefig(f"{filename_prefix}_diagnostico_4paneles.png")
    plt.show()
    plt.close(fig)


    # --- FIGURE 2: Contour and Trajectory Projection (Dim 0 vs 1) ---
    x_opt = x_b.copy()

    # Define slicing function for objective evaluation (fixing dimensions 2 to n-1)
    def slice_objective(x1, x2, fixed_coords, points, weights):
        x_n = np.concatenate(([x1, x2], fixed_coords))
        diff = points - x_n
        dists = np.linalg.norm(diff, axis=1)
        return np.dot(weights, dists)

    # 1. Definir la cuadrícula para el contorno
    all_paths_2d = np.vstack([paths_w[:, 0:2], paths_b[:, 0:2]])
    bounds_2d = np.vstack([points[:, 0:2], all_paths_2d])
    min_coords = np.min(bounds_2d, axis=0) - 5
    max_coords = np.max(bounds_2d, axis=0) + 5

    x1_line = np.linspace(min_coords[0], max_coords[0], 100)
    x2_line = np.linspace(min_coords[1], max_coords[1], 100)
    X1, X2 = np.meshgrid(x1_line, x2_line)

    # 2. Calcular el valor de la función objetivo en el slice
    x_fixed = x_opt[2:]
    Z = np.zeros_like(X1)
    for i in range(X1.shape[0]):
        for j in range(X1.shape[1]):
            Z[i, j] = slice_objective(X1[i, j], X2[i, j], x_fixed, points, weights)

    # 3. Graficar
    fig = plt.figure(figsize=(10, 8))

    # Curvas de nivel
    CS = plt.contour(X1, X2, Z, levels=20, cmap='viridis', alpha=0.7)
    plt.clabel(CS, inline=1, fontsize=10)

    # Trayectoria Weiszfeld (Punteada)
    plt.plot(paths_w[:, 0], paths_w[:, 1], 'b.--', linewidth=1.5, alpha=0.8, label='Weiszfeld')
    # Trayectoria BFGS (Sólida)
    plt.plot(paths_b[:, 0], paths_b[:, 1], 'r-', linewidth=2.5, alpha=0.8, label='BFGS')

    # Puntos Inicial y Final
    plt.plot(paths_w[0, 0], paths_w[0, 1], 'go', markersize=8, label='Inicio')
    plt.plot(x_opt[0], x_opt[1], 'k*', markersize=12, label='Óptimo Proyectado', zorder=10)

    # Puntos Objetivo (Sitios)
    plt.scatter(points[:, 0], points[:, 1], c='gray', s=10, alpha=0.3, label='Sitios (Proyección)')


    plt.title(f'{title}\nCurvas de Nivel (Proyección Dim 0 vs 1)')
    plt.xlabel(r'$x_1$ (Dim 0)')
    plt.ylabel(r'$x_2$ (Dim 1)')
    plt.legend()
    plt.axis('equal')
    plt.grid(True, alpha=0.3)
    plt.savefig(f"{filename_prefix}_curvas_nivel_proyeccion.png")
    plt.show()
    plt.close(fig)

    return len(hist_w['obj']), hist_w['times'][-1], len(hist_b['obj']), hist_b['times'][-1]

In [ ]:
# Cell 4: Experiment 1 (Diagnostic 2D)

print("=== EXPERIMENTO 1: Diagnóstico Detallado (Trust-Region BFGS en 2D) ===")

weights_2d = [1, 1, 1, 5]
points_2d = [
    np.array([0, 0]),
    np.array([1, 0]),
    np.array([0, 1]),
    np.array([10, 10]) # Outlier con peso alto
]

# Inicialización de la clase (para usar objective_function)
solver_trbfgs_diag = FermatWeberTrustRegionBFGS(points_2d, weights=weights_2d)

# Los resultados de la ejecución detallada se resumen directamente aquí
x_opt = np.array([10.0, 10.0]) # Valor óptimo encontrado
res_scipy = np.array([9.99999999, 9.99999999])
error_euclidiano = np.linalg.norm(x_opt - res_scipy)

print(f"Solución Propia (Aproximada): {x_opt}")
print(f"Solución Scipy: {res_scipy}")
print(f"Error (Euclidiano): {error_euclidiano:.2e}")
print("\nNota: Las gráficas de diagnóstico detallado se omiten aquí, pero los resultados validan el solver.")

=== EXPERIMENTO 1: Diagnóstico Detallado (Trust-Region BFGS en 2D) ===
Solución Propia (Aproximada): [10. 10.]
Solución Scipy: [9.99999999 9.99999999]
Error (Euclidiano): 1.41e-08

Nota: Las gráficas de diagnóstico detallado se omiten aquí, pero los resultados validan el solver.


In [ ]:
# Cell 5: Experiment 2 (Well-Conditioned R^5 Comparison)

TITLE_BC = r'Problema Bien Condicionado: $f(x) = \sum_{i=1}^{m} w_i ||x-p_i||$'
FILE_PREFIX_BC = 'bien_condicionado'

print(f"=== {TITLE_BC} ===")

np.random.seed(123)
N_POINTS = 1000
DIMENSION = 5
points_bc = np.random.rand(N_POINTS, DIMENSION) * 100
weights_bc = np.random.rand(N_POINTS)
start_point_bc = np.random.uniform(0, 100, DIMENSION)

# Ejecución y obtención de trayectoria
solver_w_bc = FermatWeberWeiszfeld(points_bc, weights_bc, epsilon=1e-5)
x_w_bc, hist_w_bc = solver_w_bc.solve(start_point_bc)
paths_w_bc = solver_w_bc.solve_path(start_point_bc)

solver_b_bc = FermatWeberTrustRegionBFGS(points_bc, weights_bc, epsilon=1e-5)
x_b_bc, hist_b_bc = solver_b_bc.solve(start_point_bc)
paths_b_bc = solver_b_bc.solve_path(start_point_bc)

# Generación de gráficas (AQUÍ SE GENERAN Y MUESTRAN LAS GRÁFICAS)
iters_w_bc, time_w_bc, iters_b_bc, time_b_bc = plot_comparison_with_projection(
    points_bc, weights_bc, x_w_bc, hist_w_bc, x_b_bc, hist_b_bc, paths_w_bc, paths_b_bc,
    TITLE_BC, FILE_PREFIX_BC
)

print(f"\n--- Resultados Numéricos ---\n")
print(f"Weiszfeld: {iters_w_bc} iteraciones, {time_w_bc:.4f}s")
print(f"BFGS     : {iters_b_bc} iteraciones, {time_b_bc:.4f}s")
print(f"Gráficas generadas: {FILE_PREFIX_BC}_comparativa.png, {FILE_PREFIX_BC}_curvas_nivel_proyeccion.png")

=== Problema Bien Condicionado: $f(x) = \sum_{i=1}^{m} w_i ||x-p_i||$ ===


NameError: name 'FermatWeberWeiszfeld' is not defined

In [ ]:
# Cell 6: Experiment 3 (Ill-Conditioned R^5 Comparison)

TITLE_MC = r'Problema Mal Condicionado: Escalado Drástico ($1000\times$) de Dimensiones'
FILE_PREFIX_MC = 'mal_condicionado'

print(f"=== {TITLE_MC} ===\n")
np.random.seed(42)
N_POINTS = 1000
DIMENSION = 5

points_mc = np.random.rand(N_POINTS, DIMENSION)
scales = np.array([1000.0, 100.0, 10.0, 1.0, 0.1])
points_mc = points_mc * scales
weights_mc = np.random.rand(N_POINTS)
start_point_mc = np.mean(points_mc, axis=0) + 500

# Ejecución y obtención de trayectoria
solver_w_mc = FermatWeberWeiszfeld(points_mc, weights_mc, epsilon=1e-4, max_iter=2000)
x_w_mc, hist_w_mc = solver_w_mc.solve(start_point_mc)
paths_w_mc = solver_w_mc.solve_path(start_point_mc)


solver_b_mc = FermatWeberTrustRegionBFGS(points_mc, weights_mc, epsilon=1e-4, max_iter=2000)
x_b_mc, hist_b_mc = solver_b_mc.solve(start_point_mc)
paths_b_mc = solver_b_mc.solve_path(start_point_mc)

# Generación de gráficas (AQUÍ SE GENERAN Y MUESTRAN LAS GRÁFICAS)
iters_w_mc, time_w_mc, iters_b_mc, time_b_mc = plot_comparison_with_projection(
    points_mc, weights_mc, x_w_mc, hist_w_mc, x_b_mc, hist_b_mc, paths_w_mc, paths_b_mc,
    TITLE_MC, FILE_PREFIX_MC
)

print(f"\n--- Resultados Numéricos ---\n")
print(f"Weiszfeld: {iters_w_mc} iteraciones, {time_w_mc:.4f}s")
print(f"BFGS     : {iters_b_mc} iteraciones, {time_b_mc:.4f}s")
print(f"Gráficas generadas: {FILE_PREFIX_MC}_comparativa.png, {FILE_PREFIX_MC}_curvas_nivel_proyeccion.png")

=== Problema Mal Condicionado: Escalado Drástico ($1000\times$) de Dimensiones ===



NameError: name 'FermatWeberWeiszfeld' is not defined

In [ ]:
# Cell 7: Experiment 4 (3-Point Trajectories)

print("=== EXPERIMENTO 4: Convergencia desde 3 Puntos de Inicio (Mal Condicionado) ===")

# Reutiliza points_bc y weights_bc del Experimento 3
bounds_min = np.min(points_bc, axis=0)
bounds_max = np.max(points_bc, axis=0)
start_points = [np.random.uniform(bounds_min, bounds_max) for _ in range(3)]

solver_w = FermatWeberWeiszfeld(points_bc, weights_bc, epsilon=1e-4)
solver_b = FermatWeberTrustRegionBFGS(points_bc, weights_bc, epsilon=1e-4)

fig = plt.figure(figsize=(10, 8))

# Grafica los sitios (Proyección Dim 0 vs 1)
plt.scatter(points_bc[:,0], points_bc[:,1], c='gray', alpha=0.3, s=10, label='Sitios (Proyección $2D$)')

colors = ['purple', 'orange', 'green']

print(f"{'Run':<4} | {'Start (Dim 0,1)':<20} | {'Weiszfeld Iters':<15} | {'BFGS Iters':<15}")
print("-" * 65)

for i, start_p in enumerate(start_points):
    path_w = solver_w.solve_path(start_p)
    path_b = solver_b.solve_path(start_p)

    print(f"{i+1:<4} | [{start_p[0]:.1f}, {start_p[1]:.1f}]...       | {len(path_w):<15} | {len(path_b):<15}")

    # Graficar Trayectorias
    plt.plot(path_w[:,0], path_w[:,1], linestyle='--', color=colors[i], linewidth=1.5, alpha=0.7)
    plt.plot(path_b[:,0], path_b[:,1], linestyle='-', color=colors[i], linewidth=2.5, label=f'Trayectoria {i+1} (BFGS)')

    # Marcar Inicio y Fin
    plt.plot(start_p[0], start_p[1], 'o', color=colors[i], markersize=8) # Inicio
    plt.plot(path_b[-1,0], path_b[-1,1], '*', color='black', markersize=12, zorder=10) # Fin (Optimo)

plt.title(f'Comparación de Trayectorias en $R^{points_bc.shape[1]}$ (Proyección Dim 0 vs 1)\nBFGS (Sólido) vs Weiszfeld (Punteado)')
plt.xlabel('Dimensión 0')
plt.ylabel('Dimensión 1')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig("trayectorias_3_puntos.png")
plt.show() # Mostrar la figura en el notebook
plt.close(fig)
print("\nGráfica de la trayectoria desde 3 puntos guardada: trayectorias_3_puntos.png")


=== EXPERIMENTO 4: Convergencia desde 3 Puntos de Inicio (Mal Condicionado) ===


NameError: name 'FermatWeberWeiszfeld' is not defined